# Collecte des données météorologiques – Météo-France

## Projet 5CLOU

### Contexte
Ce notebook s'inscrit dans un projet de construction d'un pipeline de données en temps réel avec Microsoft Fabric. Le projet repose sur deux sources : Météo-France pour les observations météorologiques et OpenRadiation pour les mesures de radioactivité.

### Objectif du notebook
Collecter les observations météorologiques de toutes les stations françaises à partir de l'API Météo-France, préparer les données et les rendre exploitables dans le pipeline Microsoft Fabric.

### Résultats attendus
- Récupération des données météorologiques via l'API.
- Vérification et structuration des observations.
- Préparation des données pour leur intégration dans le flux temps réel.
- Exploitation ultérieure avec les données OpenRadiation.

In [1]:
# 2.1 - IMPORTATION DES BIBLIOTHÈQUES

import requests
import json
import os

from datetime import datetime, timezone, timedelta

print("Environnement Python prêt pour la collecte météo.")

StatementMeta(, 03b784ee-0cc4-47b2-856c-b09c03aef492, 3, Finished, Available, Finished, False)

Environnement Python prêt pour la collecte météo.


In [2]:
# 3.1 - CONFIGURATION DE L'API MÉTÉO-FRANCE

# Adresse de base de l'API Package Observations v2
BASE_URL = "https://public-api.meteofrance.fr/public/DPPaquetObs/v2"

print("API configurée :", BASE_URL)

StatementMeta(, 03b784ee-0cc4-47b2-856c-b09c03aef492, 4, Finished, Available, Finished, False)

API configurée : https://public-api.meteofrance.fr/public/DPPaquetObs/v2


In [3]:
# 3.2 - VÉRIFICATION DE LA CLÉ API

API_KEY = os.environ.get("METEO_FRANCE_API_KEY")

if API_KEY:
    print("Clé API disponible dans l'environnement.")
else:
    print("Clé API non configurée.")

StatementMeta(, 03b784ee-0cc4-47b2-856c-b09c03aef492, 5, Finished, Available, Finished, False)

Clé API non configurée.


In [2]:
# 3.3 - AUTHENTIFICATION MÉTÉO-FRANCE

API_KEY = "eyJ4NXQiOiJZV0kxTTJZNE1qWTNOemsyTkRZeU5XTTRPV014TXpjek1UVmhNbU14T1RSa09ETXlOVEE0Tnc9PSIsImtpZCI6ImdhdGV3YXlfY2VydGlmaWNhdGVfYWxpYXMiLCJ0eXAiOiJKV1QiLCJhbGciOiJSUzI1NiJ9.eyJzdWIiOiJiZWF0cmljZUBjYXJib24uc3VwZXIiLCJhcHBsaWNhdGlvbiI6eyJvd25lciI6ImJlYXRyaWNlIiwidGllclF1b3RhVHlwZSI6bnVsbCwidGllciI6IlVubGltaXRlZCIsIm5hbWUiOiJEZWZhdWx0QXBwbGljYXRpb24iLCJpZCI6NTE3MTQsInV1aWQiOiIzZGFlZWQwNS1iN2Q0LTRiZjQtYTgxNC04ZjdiZTU2ZWY3ZTUifSwiaXNzIjoiaHR0cHM6XC9cL3BvcnRhaWwtYXBpLm1ldGVvZnJhbmNlLmZyOjQ0M1wvb2F1dGgyXC90b2tlbiIsInRpZXJJbmZvIjp7IjEwMFJlcVBlck1pbiI6eyJ0aWVyUXVvdGFUeXBlIjoicmVxdWVzdENvdW50IiwiZ3JhcGhRTE1heENvbXBsZXhpdHkiOjAsImdyYXBoUUxNYXhEZXB0aCI6MCwic3RvcE9uUXVvdGFSZWFjaCI6dHJ1ZSwic3Bpa2VBcnJlc3RMaW1pdCI6MCwic3Bpa2VBcnJlc3RVbml0Ijoic2VjIn19LCJrZXl0eXBlIjoiUFJPRFVDVElPTiIsInN1YnNjcmliZWRBUElzIjpbeyJzdWJzY3JpYmVyVGVuYW50RG9tYWluIjoiY2FyYm9uLnN1cGVyIiwibmFtZSI6IkRvbm5lZXNQdWJsaXF1ZXNPYnNlcnZhdGlvbiIsImNvbnRleHQiOiJcL3B1YmxpY1wvRFBPYnNcL3YyIiwicHVibGlzaGVyIjoiYmFzdGllbmciLCJ2ZXJzaW9uIjoidjIiLCJzdWJzY3JpcHRpb25UaWVyIjoiMTAwUmVxUGVyTWluIn0seyJzdWJzY3JpYmVyVGVuYW50RG9tYWluIjoiY2FyYm9uLnN1cGVyIiwibmFtZSI6IkRvbm5lZXNQdWJsaXF1ZXNQYXF1ZXRPYnNlcnZhdGlvbiIsImNvbnRleHQiOiJcL3B1YmxpY1wvRFBQYXF1ZXRPYnNcL3YyIiwicHVibGlzaGVyIjoiYmFzdGllbmciLCJ2ZXJzaW9uIjoidjIiLCJzdWJzY3JpcHRpb25UaWVyIjoiMTAwUmVxUGVyTWluIn1dLCJleHAiOjE3OTM0MDY3MTAsInRva2VuX3R5cGUiOiJhcGlLZXkiLCJpYXQiOjE3OTM0MDQ2OTgsImp0aSI6IjFkYmU3MWI4LTZhOGUtNDU0YS05YzhlLTBhZmVjMDBhODk1NyJ9.Nm23JPYjHjz7USdihMNyqfNmN72WLrilzZji1oPX4jhRN9qXvVGUkyvaz-K_qe-5JyYlQaRisKJ2nGMIHJejHL1ASseiUpLEFkcghKjgyj69r2SflNccJ8mV2E94xzhBu_9uByZeNUIKTdA-vaOc3pdQgKozGludWUyzLOyi9jeYD2qeuR-OLGIcZvKbmLPfae0dy4oAPKe4fviLsd8bdGTVcnNcSMvjf2xvpnJe-ZzFMc5vZIpjCwPAcEFtFT0EcPCdhcrpSMFo_nV46VA8d_LkCra9MNXPEredyG95V2vU_pEP3GcFUfOJgs7eqhulVqZV00ODw8f5Xs-mnOOW8A=="

headers = {
    "apikey": API_KEY
}

print("Configuration de l'authentification terminée.")

StatementMeta(, f9947cde-1101-4fbc-9f5f-cbee0bd7d534, 4, Finished, Available, Finished, False)

Configuration de l'authentification terminée.


In [3]:
# 3.4 - TEST DE CONNEXION À L'API MÉTÉO-FRANCE

url_test = BASE_URL + "/paquet/stations/infrahoraire-6m"

reponse = requests.get(
    url_test,
    headers=headers,
    timeout=30
)

print("Code HTTP :", reponse.status_code)
print("Réponse :", reponse.text[:500])

StatementMeta(, f9947cde-1101-4fbc-9f5f-cbee0bd7d534, 5, Finished, Available, Finished, False)

NameError: name 'BASE_URL' is not defined

In [4]:
# 3.5 - VÉRIFICATION DE L'EXPIRATION DE LA CLÉ

import base64
import json
from datetime import datetime, timezone

try:
    payload = API_KEY.split(".")[1]
    payload += "=" * (-len(payload) % 4)

    contenu = json.loads(base64.urlsafe_b64decode(payload))

    expiration = datetime.fromtimestamp(
        contenu["exp"],
        tz=timezone.utc
    )

    print("Expiration :", expiration)
    print("Clé expirée :", datetime.now(timezone.utc) >= expiration)

except Exception:
    print("Impossible de déterminer l'expiration de cette clé.")

StatementMeta(, f9947cde-1101-4fbc-9f5f-cbee0bd7d534, 6, Finished, Available, Finished, False)

Expiration : 2026-10-31 00:31:50+00:00
Clé expirée : True


In [7]:
# 3.6 - COLLECTE DES OBSERVATIONS MÉTÉO

# Calcul de la dernière période de 6 minutes
maintenant = datetime.now(timezone.utc)

minute = (maintenant.minute // 6) * 6

date_observation = maintenant.replace(
    minute=minute,
    second=0,
    microsecond=0
) - timedelta(minutes=6)

# Paramètres de la requête
parametres = {
    "date": date_observation.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "format": "json"
}

# Envoi de la requête
reponse = requests.get(
    url_test,
    headers=headers,
    params=parametres,
    timeout=30
)

print("Date demandée :", parametres["date"])
print("Code HTTP :", reponse.status_code)
print("Réponse :", reponse.text[:500])

StatementMeta(, 03b784ee-0cc4-47b2-856c-b09c03aef492, 9, Finished, Available, Finished, False)

Date demandée : 2026-10-30T23:54:00Z
Code HTTP : 200
Réponse : [{"lat":46.278167,"lon":5.669,"geo_id_insee":"01014002","reference_time":"2026-10-30T23:57:05Z","insert_time":"2026-10-30T23:55:02Z","validity_time":"2026-10-30T23:54:00Z","t":272.55,"td":272.25,"u":98.0,"dd":70.0,"ff":1.0,"ddraf10":60.0,"raf10":1.5,"rr_per":0.0,"t_10":null,"t_20":null,"t_50":null,"t_100":null,"vv":null,"etat_sol":null,"sss":0.0,"insolh":null,"ray_glo01":null,"pres":null,"pmer":null},{"lat":45.833,"lon":5.106667,"geo_id_insee":"01027003","reference_time":"2026-10-30T23:57:05Z","


In [8]:
# 4.1 - EXPLORATION DES DONNÉES MÉTÉOROLOGIQUES

# Conversion de la réponse JSON en données Python
donnees_meteo = reponse.json()

# Nombre total d'observations
nombre_observations = len(donnees_meteo)

print("Nombre total d'observations :", nombre_observations)

# Affichage des informations de la première observation
if nombre_observations > 0:
    premiere_station = donnees_meteo[0]

    print("\nExemple d'observation :")
    print(json.dumps(premiere_station, indent=2, ensure_ascii=False))

StatementMeta(, 03b784ee-0cc4-47b2-856c-b09c03aef492, 10, Finished, Available, Finished, False)

Nombre total d'observations : 1620

Exemple d'observation :
{
  "lat": 46.278167,
  "lon": 5.669,
  "geo_id_insee": "01014002",
  "reference_time": "2026-10-30T23:57:05Z",
  "insert_time": "2026-10-30T23:55:02Z",
  "validity_time": "2026-10-30T23:54:00Z",
  "t": 272.55,
  "td": 272.25,
  "u": 98.0,
  "dd": 70.0,
  "ff": 1.0,
  "ddraf10": 60.0,
  "raf10": 1.5,
  "rr_per": 0.0,
  "t_10": null,
  "t_20": null,
  "t_50": null,
  "t_100": null,
  "vv": null,
  "etat_sol": null,
  "sss": 0.0,
  "insolh": null,
  "ray_glo01": null,
  "pres": null,
  "pmer": null
}


In [9]:
# 4.2 - CRÉATION DU DATAFRAME PYSPARK

# Conversion des observations JSON en DataFrame
df_meteo = spark.createDataFrame(donnees_meteo)

# Affichage du nombre de lignes et de colonnes
print("Nombre de lignes :", df_meteo.count())
print("Nombre de colonnes :", len(df_meteo.columns))

# Affichage des 5 premières observations
display(df_meteo.limit(5))

StatementMeta(, 03b784ee-0cc4-47b2-856c-b09c03aef492, 11, Finished, Available, Finished, False)

Nombre de lignes : 1620
Nombre de colonnes : 25


SynapseWidget(Synapse.DataFrame, 7c11fe67-360d-4e66-9bb7-b2847615bc5c)

In [10]:
# 4.3 - CONTRÔLE DES DONNÉES MÉTÉOROLOGIQUES

from pyspark.sql import functions as F

# Nombre total d'observations
print("Nombre d'observations :", df_meteo.count())

# Nombre de stations géographiques distinctes
print(
    "Nombre de localisations distinctes :",
    df_meteo.select("lat", "lon").distinct().count()
)

# Vérification des valeurs manquantes
df_meteo.select(
    F.count(F.when(F.col("t").isNull(), 1)).alias("temperature_manquante"),
    F.count(F.when(F.col("u").isNull(), 1)).alias("humidite_manquante"),
    F.count(F.when(F.col("ff").isNull(), 1)).alias("vent_manquant")
).show()

StatementMeta(, 03b784ee-0cc4-47b2-856c-b09c03aef492, 12, Finished, Available, Finished, False)

Nombre d'observations : 1620
Nombre de localisations distinctes : 1620
+---------------------+------------------+-------------+
|temperature_manquante|humidite_manquante|vent_manquant|
+---------------------+------------------+-------------+
|                   22|               955|          957|
+---------------------+------------------+-------------+



In [11]:
# 5.4 - ENREGISTREMENT DES DONNÉES MÉTÉO DANS LE LAKEHOUSE

# Enregistrement du DataFrame au format Delta
df_meteo.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("meteo_observations")

print("Table Delta créée avec succès !")
print("Nombre d'observations enregistrées :", df_meteo.count())

StatementMeta(, 03b784ee-0cc4-47b2-856c-b09c03aef492, 13, Finished, Available, Finished, False)

Table Delta créée avec succès !
Nombre d'observations enregistrées : 1620


In [12]:
# 5.6 - VÉRIFICATION DES TABLES DU LAKEHOUSE

spark.sql("SHOW TABLES IN dbo").show(truncate=False)

StatementMeta(, 03b784ee-0cc4-47b2-856c-b09c03aef492, 14, Finished, Available, Finished, False)

+------------------------------------------+------------------+-----------+
|namespace                                 |tableName         |isTemporary|
+------------------------------------------+------------------+-----------+
|`5clou-project`.Lakehouse_meteo_France.dbo|meteo_observations|false      |
+------------------------------------------+------------------+-----------+



In [13]:
# 5.7 - VÉRIFICATION DES DONNÉES DU LAKEHOUSE

# Lire la table Delta enregistrée
df_stocke = spark.table("dbo.meteo_observations")

# Vérifier le nombre d'observations
print("Nombre d'observations stockées :", df_stocke.count())

# Afficher les 5 premières observations
display(df_stocke.limit(5))

StatementMeta(, 03b784ee-0cc4-47b2-856c-b09c03aef492, 15, Finished, Available, Finished, False)

Nombre d'observations stockées : 1620


SynapseWidget(Synapse.DataFrame, 74f9ee74-8b77-482d-a50c-56049f07a04a)

In [4]:
# 6.1 - IDENTIFICATION DES OBSERVATIONS

# Afficher les colonnes disponibles
print("Colonnes de la table :")
print(df_stocke.columns)

# Vérifier les dates d'observation
df_stocke.select(
    "geo_id_insee",
    "validity_time",
    "reference_time"
).show(10, truncate=False)

# Compter les localisations distinctes
print(
    "Localisations distinctes :",
    df_stocke.select("lat", "lon").distinct().count()
)

StatementMeta(, d2d80776-f8ed-4510-920c-716bff070c8d, 6, Finished, Available, Finished, False)

Colonnes de la table :


NameError: name 'df_stocke' is not defined

In [5]:
# 6.2 - VÉRIFICATION DES DOUBLONS

from pyspark.sql import functions as F

# Recherche des observations ayant la même position
# et la même date de validité

doublons = (
    df_stocke
    .groupBy("lat", "lon", "validity_time")
    .count()
    .filter(F.col("count") > 1)
)

print("Nombre de groupes de doublons :", doublons.count())

display(doublons.limit(10))

StatementMeta(, d2d80776-f8ed-4510-920c-716bff070c8d, 7, Finished, Available, Finished, False)

NameError: name 'df_stocke' is not defined

In [6]:
# 7.1 - LECTURE DES DONNÉES DEPUIS LE LAKEHOUSE

df_meteo = spark.table("dbo.meteo_observations")

print("Nombre d'observations :", df_meteo.count())
print("Nombre de colonnes :", len(df_meteo.columns))

# Afficher la structure des données
df_meteo.printSchema()

StatementMeta(, d2d80776-f8ed-4510-920c-716bff070c8d, 8, Finished, Available, Finished, False)

Nombre d'observations : 1620
Nombre de colonnes : 25
root
 |-- dd: double (nullable = true)
 |-- ddraf10: double (nullable = true)
 |-- etat_sol: double (nullable = true)
 |-- ff: double (nullable = true)
 |-- geo_id_insee: string (nullable = true)
 |-- insert_time: string (nullable = true)
 |-- insolh: double (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- pmer: double (nullable = true)
 |-- pres: double (nullable = true)
 |-- raf10: double (nullable = true)
 |-- ray_glo01: double (nullable = true)
 |-- reference_time: string (nullable = true)
 |-- rr_per: double (nullable = true)
 |-- sss: double (nullable = true)
 |-- t: double (nullable = true)
 |-- t_10: double (nullable = true)
 |-- t_100: double (nullable = true)
 |-- t_20: double (nullable = true)
 |-- t_50: double (nullable = true)
 |-- td: double (nullable = true)
 |-- u: double (nullable = true)
 |-- validity_time: string (nullable = true)
 |-- vv: double (nullable = true)



In [7]:
# 7.2 - PRÉPARATION DES DONNÉES MÉTÉOROLOGIQUES

from pyspark.sql import functions as F

df_meteo_prepare = (
    df_meteo
    # Conversion de la date d'observation
    .withColumn(
        "date_observation",
        F.to_timestamp("validity_time")
    )
    # Conversion de la température : Kelvin → Celsius
    .withColumn(
        "temperature_c",
        F.round(F.col("t") - 273.15, 2)
    )
    # Renommage de l'humidité pour faciliter la lecture
    .withColumn(
        "humidite_pct",
        F.col("u")
    )
)

# Afficher quelques résultats
display(
    df_meteo_prepare.select(
        "geo_id_insee",
        "date_observation",
        "temperature_c",
        "humidite_pct",
        "lat",
        "lon"
    ).limit(10)
)

StatementMeta(, d2d80776-f8ed-4510-920c-716bff070c8d, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c33c2662-9980-4c04-9caa-1e6dd9787319)

In [9]:
# 7.3 - CONTRÔLE QUALITÉ DES DONNÉES MÉTÉOROLOGIQUES

from pyspark.sql import functions as F

# Nombre total d'observations
total = df_meteo_prepare.count()

# Vérification des valeurs manquantes
controle_qualite = df_meteo_prepare.select(
    F.count(F.when(F.col("temperature_c").isNull(), 1)).alias("temperature_manquante"),
    F.count(F.when(F.col("humidite_pct").isNull(), 1)).alias("humidite_manquante"),
    F.count(F.when(F.col("date_observation").isNull(), 1)).alias("date_manquante"),
    F.count(F.when(F.col("lat").isNull() | F.col("lon").isNull(), 1)).alias("coordonnees_manquantes")
)

print("Nombre total d'observations :", total)

display(controle_qualite)

StatementMeta(, d2d80776-f8ed-4510-920c-716bff070c8d, 11, Finished, Available, Finished, False)

Nombre total d'observations : 1620


SynapseWidget(Synapse.DataFrame, 320e8cb4-26ed-443c-b48d-7313cbf1dd52)

In [10]:
# 7.4 - ENREGISTREMENT DES DONNÉES SILVER

(
    df_meteo_prepare.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dbo.meteo_observations_silver")
)

print("Table Silver créée avec succès !")

print(
    "Nombre d'observations Silver :",
    spark.table("dbo.meteo_observations_silver").count()
)

StatementMeta(, d2d80776-f8ed-4510-920c-716bff070c8d, 12, Finished, Available, Finished, False)

Table Silver créée avec succès !
Nombre d'observations Silver : 1620


In [11]:
# 7.5 - CRÉATION DES INDICATEURS MÉTÉOROLOGIQUES GOLD

from pyspark.sql import functions as F

# Lecture des données Silver
df_silver = spark.table("dbo.meteo_observations_silver")

# Calcul des indicateurs par date d'observation
df_gold = (
    df_silver
    .groupBy("date_observation")
    .agg(
        F.count("*").alias("nombre_observations"),
        F.round(F.avg("temperature_c"), 2).alias("temperature_moyenne_c"),
        F.round(F.min("temperature_c"), 2).alias("temperature_min_c"),
        F.round(F.max("temperature_c"), 2).alias("temperature_max_c"),
        F.round(F.avg("humidite_pct"), 2).alias("humidite_moyenne_pct")
    )
    .orderBy("date_observation")
)

# Affichage des résultats
display(df_gold)

StatementMeta(, d2d80776-f8ed-4510-920c-716bff070c8d, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3bbc29ee-9fe2-41ba-8391-7091fbb8d86b)

In [12]:
# 7.6 - ENREGISTREMENT DES INDICATEURS GOLD

# Enregistrer les indicateurs dans une table Delta
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dbo.meteo_observations_gold")
)

print("Table Gold créée avec succès !")

# Vérifier les données enregistrées
df_gold_stocke = spark.table("dbo.meteo_observations_gold")

print("Nombre de lignes Gold :", df_gold_stocke.count())

display(df_gold_stocke)

StatementMeta(, d2d80776-f8ed-4510-920c-716bff070c8d, 14, Finished, Available, Finished, False)

Table Gold créée avec succès !
Nombre de lignes Gold : 1


SynapseWidget(Synapse.DataFrame, f6a53013-4d5e-45b0-a70a-b4968ff3063b)

In [1]:
# 8.14 - Préparer les données météo pour Eventhouse

df_export = spark.table("dbo.meteo_observations_silver")

df_export = df_export.selectExpr(
    "date_observation",
    "cast(geo_id_insee as string) as geo_id_insee",
    "cast(lat as double) as latitude",
    "cast(lon as double) as longitude",
    "cast(temperature_c as double) as temperature_c",
    "cast(humidite_pct as double) as humidite_pct"
)

# Export dans le dossier Files du Lakehouse
(
    df_export.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv("Files/export_meteo_eventhouse")
)

print("Export météo terminé !")
print("Nombre de lignes :", df_export.count())

StatementMeta(, e5cd3948-e9d6-4906-9ac8-95570fed49b3, 3, Finished, Available, Finished, False)

Export météo terminé !
Nombre de lignes : 1620


In [2]:
# Vérifier les fichiers présents dans le Lakehouse
display(notebookutils.fs.ls("Files/"))

StatementMeta(, e5cd3948-e9d6-4906-9ac8-95570fed49b3, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a89e8d1d-7f8b-4fb6-be9e-ad065c416f14)

In [3]:
# Vérifier le Lakehouse associé au notebook

print("Lakehouse par défaut :")
print(notebookutils.runtime.context.get("defaultLakehouseName"))

print("Identifiant du Lakehouse :")
print(notebookutils.runtime.context.get("defaultLakehouseId"))

StatementMeta(, e5cd3948-e9d6-4906-9ac8-95570fed49b3, 5, Finished, Available, Finished, False)

Lakehouse par défaut :
Lakehouse_meteo_France
Identifiant du Lakehouse :
947996d1-ebad-4fcd-8948-b388ed27d6e3


In [4]:
# Vérifier si le dossier d'exportation existe

chemin = "Files/export_meteo_eventhouse"

print("Dossier présent :", notebookutils.fs.exists(chemin))

StatementMeta(, e5cd3948-e9d6-4906-9ac8-95570fed49b3, 6, Finished, Available, Finished, False)

Dossier présent : True


In [5]:
# 8.19 - Afficher les fichiers exportés

fichiers = notebookutils.fs.ls("Files/export_meteo_eventhouse")

for fichier in fichiers:
    print(fichier.name, "| Taille :", fichier.size, "octets")

StatementMeta(, e5cd3948-e9d6-4906-9ac8-95570fed49b3, 7, Finished, Available, Finished, False)

_SUCCESS | Taille : 0 octets
part-00000-11243db6-360e-4922-bcb1-8c717aa371d2-c000.csv | Taille : 95169 octets


In [6]:
# 8.20 - Récupérer le chemin exact du fichier CSV

fichiers = notebookutils.fs.ls("Files/export_meteo_eventhouse")

for fichier in fichiers:
    if fichier.name.endswith(".csv"):
        print("Nom du fichier :", fichier.name)
        print("Chemin complet :", fichier.path)

StatementMeta(, e5cd3948-e9d6-4906-9ac8-95570fed49b3, 8, Finished, Available, Finished, False)

Nom du fichier : part-00000-11243db6-360e-4922-bcb1-8c717aa371d2-c000.csv
Chemin complet : abfss://0bcbec59-dbbb-41e1-a6a0-d3763e6c43e6@onelake.dfs.fabric.microsoft.com/947996d1-ebad-4fcd-8948-b388ed27d6e3/Files/export_meteo_eventhouse/part-00000-11243db6-360e-4922-bcb1-8c717aa371d2-c000.csv
